In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 1416, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 1416 (delta 22), reused 31 (delta 12), pack-reused 1350 (from 3)
Receiving objects: 100% (1416/1416), 375.21 MiB | 19.59 MiB/s, done.
Resolving deltas: 100% (757/757), done.
Updating files: 100% (144/144), done.


In [3]:
%cd capstone_project_GroupA
!git checkout main

/content/capstone_project_GroupA
Already on 'main'
Your branch is up to date with 'origin/main'.


In [4]:
%cd src

/content/capstone_project_GroupA/src


In [5]:
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/capstone_project_GroupA/patchtst_tuning'

In [6]:
from datetime import datetime
from ModelFiles.GroupAModels import TransformersModel
from ModelFiles.ModelConfigs import TransformersConfig, HORIZONS, SEEDS
from ModelFiles.ModelEnums import TransformerModelType
from ModelFiles.ModelPlots import *

USE_LOG_TARGET = True
CONTEXT_LENGTHS = [336, 720] # 336 is original. However, the authors used hourly data and we are using half hourly data. so try 720 as well.
EVAL_STEP_SIZE = 48
NUM_EPOCHS = 100
PATIENCE = 10
DEBUG = False
SEEDS = [31415]
PARAMETERS = [(128, 256, 3, 0.2, 0.0001), (128, 256, 3, 0.0, 0.000005), (512, 2048, 5, 0.2, 0.0001), (512, 2048, 5, 0.0, 0.000005)]
for horizon in HORIZONS:
    for context_length in CONTEXT_LENGTHS:
        for seed in SEEDS:
            for d_model, dim_ff, num_encoder_layers, dropout, learning_rate in PARAMETERS:
                patchtst_config = TransformersConfig(
                    task_id=f"patchtst_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                    model=TransformerModelType.PATCHTST,
                    forecast_horizon=horizon,
                    lookback_window=context_length,
                    used_log_target=USE_LOG_TARGET,
                    target_col= "LOG_TOTALDEMAND" if USE_LOG_TARGET else "TOTALDEMAND",
                    feature_cols=['TEMPERATURE', 'TEMP_SQUARED', 'IS_WEEKEND', 'demand_1_year_ago'],
                    scale=True,
                    date_col='DATETIME',
                    variate='MS',
                    patch_len=16,
                    stride=8,
                    d_model=d_model,
                    num_attention_heads=16,
                    num_encoder_layers=num_encoder_layers,
                    dim_ff=dim_ff,
                    dropout=dropout,
                    dropout_head_fc=dropout,
                    use_gpu=True,
                    time_encoding='timeF',
                    training_epochs=NUM_EPOCHS,
                    batch_size=32,
                    learning_rate=learning_rate,
                    output_attention=False,
                    lradj='TST',
                    patience=PATIENCE,
                    seed=seed,
                    eval_step_size=EVAL_STEP_SIZE,
                    save_test_results=False,
                    debug=DEBUG,
                    save_training_log=True,
                    save_model=False,
                )
                patch_tst_model = TransformersModel(patchtst_config, specific_output_dir=SAVE_PATH)
                patch_tst_model.train_model()
                print("=" * 200)
                print("\n")

Found NSW data path: /content/capstone_project_GroupA/data/NSW
Saved files to: /content/capstone_project_GroupA/data/NSW
['test.csv', 'train_scaled.csv', 'forecastdemand_nsw.csv.zip.partaa', 'test_scaled.csv', 'val_scaled.csv', 'validation.csv', 'forecastdemand_nsw.csv.zip', 'train.csv', 'temperature_nsw.csv.zip', 'forecastdemand_nsw.csv.zip.partab', 'totaldemand_nsw.csv.zip', 'df_model.csv', 'scaler.pkl']
Set random seed to 31415
Use GPU: cuda:0
train 124912
	iters: 100, epoch: 1 | loss: 0.6263034
	speed: 0.0950s/iter; left time: 37066.9888s
	iters: 200, epoch: 1 | loss: 0.5181692
	speed: 0.0147s/iter; left time: 5748.3550s
	iters: 300, epoch: 1 | loss: 0.5232657
	speed: 0.0150s/iter; left time: 5839.6901s
	iters: 400, epoch: 1 | loss: 0.4234793
	speed: 0.0145s/iter; left time: 5642.6689s
	iters: 500, epoch: 1 | loss: 0.4043225
	speed: 0.0145s/iter; left time: 5666.6117s
	iters: 600, epoch: 1 | loss: 0.4269894
	speed: 0.0146s/iter; left time: 5701.1744s
	iters: 700, epoch: 1 | loss: 0

KeyboardInterrupt: 